# ДЗ №4. Data Understanding и EDA

Финальный EDA по `data/processed/news_risk_dataset_labeled.csv`.

## 1. Источник данных

Источник — `IlyaGusev/ru_news`. Полный RuNews не сохраняется в репозитории. Единица наблюдения — фрагмент новости вокруг найденного банка.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_colwidth', 160)
root = Path.cwd()
if not (root / 'data').exists() and (root.parent / 'data').exists():
    root = root.parent
path = root / 'data/processed/news_risk_dataset_labeled.csv'
df = pd.read_csv(path, keep_default_na=False)
df.shape, df.head()

## 2. Состав датасета

In [ ]:
display(pd.DataFrame({'column': df.columns}))
display(df[['sample_id', 'dataset_part', 'entity_norm', 'risk_type', 'alert_flag', 'split']].head())

## 3. Распределения

In [ ]:
for column in ['alert_flag', 'entity_relevance', 'entity_norm', 'source', 'published_year', 'split']:
    counts = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name='count')
    display(counts.head(20))
    counts.head(20).set_index(column)['count'].sort_values().plot(kind='barh', figsize=(8, 4), title=column)
    plt.tight_layout()
    plt.show()

### Техническая проверка dataset_part

Проверяем, из каких частей собран датасет. Это технический признак происхождения строки, а не целевая переменная для обучения.

In [ ]:
dataset_part_counts = df['dataset_part'].value_counts().rename_axis('dataset_part').reset_index(name='count')
display(dataset_part_counts)
display(pd.crosstab(df['dataset_part'], df['alert_flag']))

## 3.1. Risk Type: полный график и редкие классы

`no_risk` ожидаемо доминирует, поэтому дополнительно смотрим риск-классы без `no_risk` и логарифмическую шкалу.

In [ ]:
risk_counts = df['risk_type'].value_counts().rename_axis('risk_type').reset_index(name='count')
display(risk_counts)
risk_counts.set_index('risk_type')['count'].sort_values().plot(kind='barh', figsize=(8, 4), title='Распределение risk_type: полный датасет')
plt.tight_layout(); plt.show()

risk_only = risk_counts[risk_counts['risk_type'] != 'no_risk']
display(risk_only)
risk_only.set_index('risk_type')['count'].sort_values().plot(kind='barh', figsize=(8, 4), title='Распределение риск-классов без no_risk')
plt.tight_layout(); plt.show()

risk_counts.set_index('risk_type')['count'].sort_values().plot(kind='barh', logx=True, figsize=(8, 4), title='Распределение risk_type в логарифмической шкале')
plt.tight_layout(); plt.show()

## 3.2. Alert Flag по частям датасета

In [ ]:
alert_counts = df['alert_flag'].value_counts().rename_axis('alert_flag').reset_index(name='count')
display(alert_counts)
alert_counts.set_index('alert_flag')['count'].sort_values().plot(kind='barh', figsize=(7, 3), title='Бинарная целевая метка alert_flag')
plt.tight_layout(); plt.show()

part_alert = pd.crosstab(df['dataset_part'], df['alert_flag'])
display(part_alert)
part_alert.plot(kind='bar', figsize=(9, 4), title='Распределение alert_flag по dataset_part')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 4. Длины, пропуски и дубли

In [ ]:
display(df['text_fragment'].astype(str).str.len().describe().rename('text_fragment_length'))
df['text_fragment'].astype(str).str.len().hist(bins=30, figsize=(8, 4))
plt.title('Длина text_fragment')
plt.tight_layout()
plt.show()

missing = df.astype(str).apply(lambda col: col.str.strip().eq('')).sum().rename('missing_count').to_frame()
display(missing)
print('Дубли url + entity_norm + text_fragment:', df.duplicated(['url', 'entity_norm', 'text_fragment']).sum())

## Текстовые характеристики

Анализируем длины текстов, риск-слова и частотные слова для будущей NLP baseline-модели.

In [ ]:
import re
from collections import Counter

STOP_WORDS = {'и','в','во','на','с','со','к','ко','по','из','за','от','до','для','что','как','это','или','но','а','о','об','у','не','он','она','они','мы','вы','его','ее','их','был','была','были','будет','также'}

def word_count(series):
    return series.fillna('').astype(str).str.findall(r'[A-Za-zА-Яа-яЁё]+').str.len()

length_rows = []
for column in ['title', 'text_fragment']:
    chars = df[column].astype(str).str.len()
    words = word_count(df[column])
    length_rows.append({
        'column': column,
        'char_mean': chars.mean(), 'char_median': chars.median(), 'char_min': chars.min(), 'char_max': chars.max(), 'char_p25': chars.quantile(.25), 'char_p75': chars.quantile(.75),
        'word_mean': words.mean(), 'word_median': words.median(), 'word_min': words.min(), 'word_max': words.max(), 'word_p25': words.quantile(.25), 'word_p75': words.quantile(.75),
    })
display(pd.DataFrame(length_rows))

df['text_fragment'].astype(str).str.len().hist(bins=30, figsize=(8, 4))
plt.title('Распределение длины text_fragment')
plt.tight_layout(); plt.show()

In [ ]:
tmp = df.assign(text_fragment_chars=df['text_fragment'].astype(str).str.len())
length_by_risk = tmp.groupby('risk_type')['text_fragment_chars'].describe().sort_values('count', ascending=False)
display(length_by_risk)
length_by_risk['50%'].sort_values().plot(kind='barh', figsize=(8, 4), title='Медианная длина text_fragment по risk_type')
plt.tight_layout(); plt.show()

In [ ]:
keyword_counter = Counter()
for value in df['found_risk_keywords'].fillna('').astype(str):
    for part in re.split(r'[;,|]', value):
        keyword = part.strip().lower()
        if keyword:
            keyword_counter[keyword] += 1
risk_keywords_top = pd.DataFrame(keyword_counter.most_common(30), columns=['keyword', 'count'])
display(risk_keywords_top)
if not risk_keywords_top.empty:
    risk_keywords_top.set_index('keyword')['count'].sort_values().plot(kind='barh', figsize=(8, 6), title='Топ риск-слов')
    plt.tight_layout(); plt.show()

In [ ]:
def tokenize(text):
    tokens = re.findall(r'[A-Za-zА-Яа-яЁё]+', str(text).lower())
    return [token for token in tokens if len(token) >= 3 and token not in STOP_WORDS]

def top_words(series, n=30):
    counter = Counter()
    for text in series:
        counter.update(tokenize(text))
    return pd.DataFrame(counter.most_common(n), columns=['word', 'count'])

for label, subset in [('alert_flag = 1', df[df['alert_flag'].astype(str) == '1']), ('alert_flag = 0', df[df['alert_flag'].astype(str) == '0'])]:
    words = top_words(subset['text_fragment'])
    print(label)
    display(words)
    words.set_index('word')['count'].sort_values().plot(kind='barh', figsize=(8, 6), title=f'Топ слов: {label}')
    plt.tight_layout(); plt.show()

Выводы для моделирования:

- Фрагменты имеют ограниченную длину, поэтому подходят для TF-IDF baseline.
- Сильный дисбаланс классов остается основной проблемой.
- Частотные слова и риск-слова помогают объяснить rule-based baseline.
- Для более сильной модели нужно сравнить TF-IDF baseline с эмбеддингами / RuBERT.

## 5. Выводы для моделирования

- Датасет пригоден для первой baseline-модели.
- Основная задача обучения — бинарная классификация `alert_flag`.
- `risk_type` сохраняется как дополнительная метка.
- Из-за дисбаланса классов для обучения нужен balanced train.
- `valid` и `test` лучше оставить ближе к исходному распределению.